In [ ]:
# -*- coding: utf-8 -*-
# ===============================================
# FACE RECOGNITION PIPELINE (MTCNN + FaceNet + SVM)
# ===============================================

# 1️⃣ Install dependencies (Run once in Jupyter terminal if needed)
# pip install mtcnn keras-facenet opencv-python matplotlib scikit-learn

# 2️⃣ Import Libraries
import cv2 as cv
import os
import numpy as np
import matplotlib.pyplot as plt
from mtcnn.mtcnn import MTCNN
from keras_facenet import FaceNet
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import pickle

# Ignore TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# =============================
# 3️⃣ Sample Face Detection
# =============================
image_path = "../dataset/test/sinner/sinner.jpeg"
img = cv.imread(image_path)
img = cv.cvtColor(img, cv.COLOR_BGR2RGB)

detector = MTCNN()
results = detector.detect_faces(img)

x, y, w, h = results[0]['box']
cv.rectangle(img, (x,y), (x+w, y+h), (0,0,255), 3)
plt.imshow(img)
plt.axis('off')
plt.show()

# Crop face and resize to 160x160
my_face = img[y:y+h, x:x+w]
my_face = cv.resize(my_face, (160,160))
plt.imshow(my_face)
plt.axis('off')
plt.show()

# =============================
# 4️⃣ Face Dataset Loader Class
# =============================
class FACELOADING:
    def __init__(self, directory):
        self.directory = directory
        self.target_size = (160,160)
        self.X = []
        self.Y = []
        self.detector = MTCNN()

    def extract_face(self, filename):
        img = cv.imread(filename)
        if img is None:
            return None
        img = cv.cvtColor(img, cv.COLOR_BGR2RGB)
        results = self.detector.detect_faces(img)
        if len(results) == 0:
            return None
        x, y, w, h = results[0]['box']
        x, y = abs(x), abs(y)
        face = img[y:y+h, x:x+w]
        face_arr = cv.resize(face, self.target_size)
        return face_arr

    def load_faces(self, dir):
        FACES = []
        for im_name in os.listdir(dir):
            try:
                path = os.path.join(dir, im_name)
                single_face = self.extract_face(path)
                if single_face is not None:
                    FACES.append(single_face)
            except Exception as e:
                pass
        return FACES

    def load_classes(self):
        for sub_dir in os.listdir(self.directory):
            path = os.path.join(self.directory, sub_dir)
            FACES = self.load_faces(path)
            labels = [sub_dir for _ in range(len(FACES))]
            print(f"Loaded successfully: {sub_dir} ({len(labels)} faces)")
            self.X.extend(FACES)
            self.Y.extend(labels)
        return np.asarray(self.X), np.asarray(self.Y)

# =============================
# 5️⃣ Load Dataset
# =============================
train_dir = "../dataset/train"  # Update your dataset path here
faceloading = FACELOADING(train_dir)
X, Y = faceloading.load_classes()

print("Total Faces Loaded:", len(X))
print("Class Labels:", np.unique(Y))

# =============================
# 6️⃣ Face Embeddings using FaceNet
# =============================
embedder = FaceNet()

def get_embedding(face_img):
    face_img = face_img.astype('float32')
    face_img = np.expand_dims(face_img, axis=0)
    yhat = embedder.embeddings(face_img)
    return yhat[0]

EMBEDDED_X = []
for img in X:
    EMBEDDED_X.append(get_embedding(img))

EMBEDDED_X = np.asarray(EMBEDDED_X)

# Save embeddings for later use
np.savez_compressed('faces_embeddings_done_4classes.npz', EMBEDDED_X, Y)
print("✅ Embeddings saved successfully!")

# =============================
# 7️⃣ Encode Labels and Split Dataset
# =============================
encoder = LabelEncoder()
encoder.fit(Y)
Y_enc = encoder.transform(Y)

X_train, X_test, Y_train, Y_test = train_test_split(
    EMBEDDED_X, Y_enc, shuffle=True, random_state=17, test_size=0.2)

print("Train size:", len(X_train), "Test size:", len(X_test))

# =============================
# 8️⃣ Train SVM Model
# =============================
model = SVC(kernel='linear', probability=True)
model.fit(X_train, Y_train)

ypreds_train = model.predict(X_train)
ypreds_test = model.predict(X_test)

print("✅ Train Accuracy:", accuracy_score(Y_train, ypreds_train))
print("✅ Test Accuracy:", accuracy_score(Y_test, ypreds_test))

# =============================
# 9️⃣ Test on New Image
# =============================
test_img_path = "img_dataset/test/bellingham/bellingham.jpg"  # Update your path
t_im = cv.imread(test_img_path)
t_im = cv.cvtColor(t_im, cv.COLOR_BGR2RGB)

result = detector.detect_faces(t_im)
if len(result) > 0:
    x,y,w,h = result[0]['box']
    t_im = t_im[y:y+h, x:x+w]
    t_im = cv.resize(t_im, (160,160))
    test_im = get_embedding(t_im)

    ypred = model.predict([test_im])
    name = encoder.inverse_transform(ypred)
    print("✅ Predicted Person:", name[0])
    plt.imshow(t_im)
    plt.title(f"Predicted: {name[0]}")
    plt.axis('off')
    plt.show()
else:
    print("❌ No face detected in test image!")

# =============================
# 🔟 Save the SVM Model
# =============================
with open('svm_model_160x160.pkl', 'wb') as f:
    pickle.dump(model, f)

print("✅ SVM model saved successfully!")